---
---
# **Tutorial 3B:** *Model Context Protocol (MCP)*
### *Turning our formulary tools into a service*
---
---

### QUESTIONS FOR TODAY
> 1. *Last time we already made tool calling work. So what problem is MCP actually solving?*
> 2. *If our AI team builds a formulary tool, how can a different department (like pharmacy operations) use it without copying our code?*

### THE ONE IDEA
> **MCP does not change what a tool is. It standardises how tools are described, discovered and called - so a tool your team builds becomes a service other teams can use.**

### OUR ROADMAP
| Step | What we do |
|---|---|
| 1 | **Calling Tools through MCP** - being the Server |
| 2 | **Connect MCP to our local model** - the full loop |
| 3 | **Why a standard matters** - for regulated environments especially |
| 4 | **Using real world MCP Server** - being the Client |

---
# **Step 0: Setting Up**
---

Ollama running locally (as in previous sessions) plus the `mcp` package.

In [ ]:
# ---------------------------------------------------------
# 1. INSTALL THE MCP SDK
# ---------------------------------------------------------
%pip install -q mcp

# ---------------------------------------------------------
# 2. BRING IN OUR TOOLKITS
# ---------------------------------------------------------
import json
import pandas as pd
import requests
from mcp.server.mcpserver import MCPServer      # MCP 2.x

# ---------------------------------------------------------
# 3. POINT AT THE LOCAL MODEL
# ---------------------------------------------------------
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen3.5:4b"      # 3.4 GB, supports tool calling

# ---------------------------------------------------------
# 4. LOAD OUR DATA
# ---------------------------------------------------------
formulary    = pd.read_csv("data/drug_formulary.csv")       # from last session
interactions = pd.read_csv("data/drug_interactions.csv")    # new today

print("Formulary drugs      :", len(formulary))
print("Known interactions   :", len(interactions))

In [ ]:
# ---------------------------------------------------------
# THE NEW DATA: DRUG INTERACTIONS
# ---------------------------------------------------------
interactions[["drug_a", "drug_b", "severity", "advice"]]

Note the one **Major** interaction in there: *Nimesulide + Paracetamol*. This is exactly the kind of check that must come from a **data source**, never from a model's memory.

---
# **Step 1: Calling Tools through MCP**
---

We simply **decorate** the python functions/tools, and MCP reads the function itself — name, type hints, docstring — to build the schema.

In [ ]:
# ---------------------------------------------------------
# CREATE AN MCP SERVER
# ---------------------------------------------------------
pharmacy = MCPServer("Formulary Assistant")


# ---------------------------------------------------------
# THE TOOLS
# ---------------------------------------------------------
@pharmacy.tool()
def get_drug_price(drug_name: str) -> str:
    """Look up the price of a drug in our formulary."""
    row = formulary[formulary["drug_name"].str.lower() == drug_name.lower().strip()]
    if row.empty:
        return f"'{drug_name}' is not in our formulary."
    d = row.iloc[0]
    if d["price_inr_per_strip"] == 0:
        return f"{d['drug_name']} is not marketed (India status: {d['india_status']})."
    return f"{d['drug_name']}: Rs {d['price_inr_per_strip']} per {d['pack_size']}"


@pharmacy.tool()
def check_drug_status(drug_name: str, country: str) -> str:
    """Check whether a drug is approved, restricted or banned in a given country.
    Use for any question about legality, bans or regulatory status."""
    row = formulary[formulary["drug_name"].str.lower() == drug_name.lower().strip()]
    if row.empty:
        return f"'{drug_name}' is not in our formulary."
    columns = {"india": ("india_status", "India"), "us": ("us_status", "USA"),
               "usa": ("us_status", "USA"), "united states": ("us_status", "USA"),
               "uk": ("uk_status", "UK"), "united kingdom": ("uk_status", "UK")}
    lookup = columns.get(country.lower().strip())
    if lookup is None:
        return f"We only hold data for India, USA and UK - not '{country}'."
    column, label = lookup
    d = row.iloc[0]
    return f"{d['drug_name']} in {label}: {d[column]}. Note: {d['regulatory_note']}"


@pharmacy.tool()
def check_interaction(drug_a: str, drug_b: str) -> str:
    """Check whether two drugs interact when taken together, and how serious it is."""
    a, b = drug_a.lower().strip(), drug_b.lower().strip()

    # interactions are symmetric - check the pair in both orders
    match = interactions[
        ((interactions["drug_a"].str.lower() == a) & (interactions["drug_b"].str.lower() == b)) |
        ((interactions["drug_a"].str.lower() == b) & (interactions["drug_b"].str.lower() == a))
    ]

    if match.empty:
        return (f"No interaction on record between {drug_a} and {drug_b}. "
                f"Note: absence from this file does not prove the pair is safe.")

    i = match.iloc[0]
    return (f"{i['severity'].upper()} interaction between {i['drug_a']} and {i['drug_b']}: "
            f"{i['effect']}. Advice: {i['advice']}")


print("Three tools registered on the MCP server. No JSON written by us.")

👀 Notice too that `check_interaction` returns a careful message when it finds nothing: *"absence from this file does not prove the pair is safe."*

**The model reads that sentence** — in a clinical context, honest tool output is a safety feature.

## `list_tools()` is how *any* MCP client discovers what a server can do. Let's ask our own server.

In [ ]:
# ---------------------------------------------------------
# ASK THE SERVER WHAT IT CAN DO
# ---------------------------------------------------------
# Note the 'await' - MCP is asynchronous. In a notebook you can
# use await directly at the top level like this.
tools = await pharmacy.list_tools()

print("=== MCP AUTO-GENERATED SCHEMAS ===\n")
for t in tools:
    print(f"Tool        : {t.name}")
    print(f"Description : {t.description}")           # <- from the docstring
    print(f"Input schema:")
    print(json.dumps(t.input_schema, indent=4))       # <- from the type hints
    print()

## `call_tool(name, arguments)` — the standard way any client asks any server to run something.

In [ ]:
# ---------------------------------------------------------
# A SMALL HELPER TO READ MCP RESULTS
# ---------------------------------------------------------
# MCP returns a structured result object rather than a bare value,
# because a tool can return text, images, or several items.
def read_result(result):
    """Turn an MCP CallToolResult into a plain Python value."""
    if result.structured_content:                    # simple values arrive wrapped
        return result.structured_content.get("result", result.structured_content)
    texts = [block.text for block in result.content]
    return texts[0] if len(texts) == 1 else texts


# ---------------------------------------------------------
# CALL TOOLS THE MCP WAY
# ---------------------------------------------------------
r1 = await pharmacy.call_tool("check_interaction",
                              {"drug_a": "Nimesulide", "drug_b": "Paracetamol"})
print(read_result(r1), "\n")

r2 = await pharmacy.call_tool("check_drug_status",
                              {"drug_name": "Nimesulide", "country": "India"})
print(read_result(r2), "\n")

r3 = await pharmacy.call_tool("get_drug_price", {"drug_name": "Metformin"})
print(read_result(r3), "\n")

# a pair we have no record of
r4 = await pharmacy.call_tool("check_interaction",
                              {"drug_a": "Metformin", "drug_b": "Amoxicillin"})
print(read_result(r4))

---
# **Step 2: Connecting MCP to Our Local Model**
---

Our local model speaks Ollama's tool format, not MCP's. One small translation — and crucially, **we build the model's tool list from the server itself** rather than maintaining a second copy by hand.

In [ ]:
# ---------------------------------------------------------
# TRANSLATE MCP SCHEMAS -> OLLAMA'S TOOL FORMAT
# ---------------------------------------------------------
mcp_tools = await pharmacy.list_tools()

ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description,
            "parameters": t.input_schema,
        },
    }
    for t in mcp_tools
]

print("Tools handed to the model:", [t["function"]["name"] for t in ollama_tools])

In [ ]:
# ---------------------------------------------------------
# THE FULL LOOP: 
# ASK -> MODEL REQUESTS -> MCP RUNS -> MODEL ANSWERS
# ---------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a formulary assistant for a health services company. Always use the "
    "tools to look up drug prices, regulatory status and interactions. "
    "Never guess or rely on your own knowledge about medicines."
)


async def ask(question):
    """Same 4-step handshake as last session - but MCP does the dispatching."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    # ===================== Step 1: ask, with the tool menu (built from MCP) =====================
    r = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME, "messages": messages,
        "tools": ollama_tools, "stream": False,
        "think": False,          # keeps the local model fast
    }).json()

    message = r["message"]

    # If no tool was wanted, it already answered
    if "tool_calls" not in message:
        print("----------(No tool needed)----------")
        return message.get("content", "")

    # ===================== Step 2: read the request =====================
    fc = message["tool_calls"][0]["function"]
    print(f"----------Model requested : {fc['name']}({fc['arguments']})----------")

    # ===================== Step 3: DISPATCH THROUGH MCP (no registry of ours!) =====================
    mcp_result = await pharmacy.call_tool(fc["name"], fc["arguments"])
    result = read_result(mcp_result)
    print(f"----------MCP returned    : {result}----------")

    # ===================== Step 4: send the result back =====================
    messages += [message, {"role": "tool", "content": str(result)}]

    final = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME, "messages": messages,
        "tools": ollama_tools, "stream": False, "think": False,
    }).json()

    print(f"\n----------FINAL MESSAGE----------")

    return final["message"]["content"]

In [ ]:
# ---------------------------------------------------------
# TEST 1: the safety question
# ---------------------------------------------------------
print(await ask("Is it safe to take Nimesulide together with Paracetamol?"))

In [ ]:
# ---------------------------------------------------------
# TEST 2: a regulatory question (same code, different tool chosen)
# ---------------------------------------------------------
print(await ask("Is Nimesulide allowed in India?"))

👀 Same four-step handshake from last session.

| | Last session (manual) | Today (MCP) |
|---|---|---|
| Describe a tool | write JSON by hand | `@server.tool()` decorator |
| Keep a registry | `TOOL_REGISTRY = {...}` | the server *is* the registry |
| Dispatch a call | `TOOL_REGISTRY[name](**args)` | `await server.call_tool(name, args)` |
| Discover tools | you already know them | `await server.list_tools()` |
| Another team uses it | copy-paste your code | connect to the server |

---
# **Step 3: Why This Standard Matters in Healthcare and Regulated Environments**
---

In a clinical or enterprise setting, a tool is never just a piece of code—it is an **auditable decision point**. 

* **The Single Source of Truth:** If a regulatory rule changes (e.g., a drug's restriction status updates in the formulary database), updating the underlying Python function automatically propagates the correct schema to every consumer via MCP. There is zero risk of the pharmacy claims team running an outdated JSON schema while your clinical team runs a newer one.


* **Centralized Auditing & Security:** When your tools live behind a standard MCP server, you gain a single gatekeeper where you can log every single tool invocation, track data access policies, and monitor latency or errors across all consuming applications (Claude Desktop, internal dashboards, or department agents) without modifying their code.

## SIMULATING A SEPARATE CONSUMER APP VIA STANDARD MCP
---------------------------------------------------------


Let's simulate a **downstream consumer** (like a pharmacy operations app) interacting with our server *only* via the standard MCP protocol *without importing any of our source code or functions*.

Imagine this code runs in an entirely different repository or team's app.
They don't import 'formulary' or our functions; they only talk to the 'pharmacy' MCP server.

```python
  async def consumer_client_simulation():
      print("--- [Consumer App] Discovering available tools via MCP ---")
      available_tools = await pharmacy.list_tools()
      tool_names = [t.name for t in available_tools]
      print(f"Discovered tools on server: {tool_names}\n")

      print("--- [Consumer App] Executing tool remotely via standard protocol ---")
      response = await pharmacy.call_tool("check_interaction", {"drug_a": "Nimesulide", "drug_b": "Paracetamol"})
      print(f"Result received by consumer app: {read_result(response)}")

  # Run the decoupled consumer loop
  await consumer_client_simulation()
```

---
# **Step 4: Using Somebody Else's MCP Server** 🌐
---

Everything so far, we wrote ourselves. 
> But the whole promise of a standard is that **you can use servers you did not write** — and that other teams can use yours.

So let's actually do it. We will connect to a **real, public MCP server**: the official **fetch** server, which retrieves a web page and converts it to clean markdown.

Why this matters for us?
> A formulary team constantly needs to read **published regulatory pages** — a CDSCO notice, a drug label, a guidance document. Nobody on your team has to write a scraper for that. Somebody already published a server that does it.

👀 Browse what else exists: [mcpservers.org](https://mcpservers.org/) and the official [registry.modelcontextprotocol.io](https://registry.modelcontextprotocol.io/)

### First - one important practical detail ⚠️

The fetch server has **its own dependencies**, and they conflict with ours (it needs `mcp` 1.x, our notebook uses 2.x). If we `pip install` it here, **we would break this notebook.** The solution is also the correct architecture: **an MCP server runs as its own separate process, with its own dependencies.** We never import it — we *talk* to it. The standard tool for this is `uvx`, which downloads and runs a package in a throwaway isolated environment:

```bash
pip install uv          # one time
uvx mcp-server-fetch --help    # downloads + runs it, isolated
```

In [ ]:
# ---------------------------------------------------------
# INSTALL uv (one time) - lets us run MCP servers in isolation
# ---------------------------------------------------------
%pip install -q uv

# Warm up the fetch server so the first real call is not a slow download.
# (Run this once; it caches.)
!uvx mcp-server-fetch --help > /dev/null 2>&1 && echo "fetch server ready" || echo "check that uv is on your PATH"

## NOW CONNECT TO AN EXTERNAL MCP SERVER AS A *CLIENT* 

Until now we have only been on the **server** side, defining tools. To use somebody else's server we switch hats and become an **MCP client** — the same role Claude Desktop or VS Code plays when it connects to a server. Server from : https://mcpservers.org/servers/modelcontextprotocol/fetch

In [ ]:
# ---------------------------------------------------------
# CONNECT
# ---------------------------------------------------------
import warnings, logging
warnings.filterwarnings("ignore")      # quiet some version-mismatch chatter
logging.disable(logging.WARNING)

import os
from mcp import Client, StdioServerParameters

# How to launch the server: a command, exactly like typing it in a terminal.
# 'stdio' transport = the server is a subprocess we talk to over its
# standard input/output. This is the normal way to run a LOCAL MCP server.
FETCH_SERVER = StdioServerParameters(
    command="uvx",
    args=["mcp-server-fetch"],

    # IMPORTANT on a corporate network: by default MCP passes only a
    # minimal environment to the server process, which drops proxy
    # settings. Passing our environment through lets it reach the internet.
    env={**os.environ},
)

# ---------------------------------------------------------
# CONNECT AND ASK THE SERVER WHAT IT CAN DO  (the discovery half of MCP)
# ---------------------------------------------------------
async with Client(FETCH_SERVER) as client:
    print("Connected to:", client.server_info.name)

    listing = await client.list_tools()
    for t in listing.tools:
        print("\n Tool       :", t.name)
        print(" Description:", (t.description or "").split("\n")[0])
        print(" Parameters :", list(t.input_schema.get("properties", {})))
        print(" Required   :", t.input_schema.get("required"))

> **We did not read this server's documentation. We did not import its code.**

We asked it `list_tools()` and it told us: it has a tool called `fetch`, it takes a `url` (required) plus some optional parameters.

## *FETCH* A REAL PUBLISHED DOCUMENT THROUGH THE EXTERNAL SERVER

In [ ]:
# ---------------------------------------------------------
# SCRAPE A PAGE AND SAVE IT LOCALLY
# ---------------------------------------------------------
# A public drug-information page - the kind a formulary team reads.
URL = "https://www.nhs.uk/medicines/paracetamol-for-adults/"

async with Client(FETCH_SERVER) as client:
    result = await client.call_tool(
        "fetch",
        {"url": URL, "max_length": 1500},   # keep it short for the demo
    )
    document = result.content[0].text

print("Characters fetched:", len(document))
print("=" * 60)

# The server returns a readable message instead of crashing if a site
# refuses - so check what we actually got back.
if "robots.txt" in document or len(document) < 200:
    print("The site refused automated fetching, or the network blocked it.")
    print("Server said:", document[:200])
else:
    # --- save it to our own machine ---
    os.makedirs("downloads", exist_ok=True)
    file_path = "downloads/scraped_page.md"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(document)

    print("Saved to      :", file_path)
    print("File size     :", os.path.getsize(file_path), "bytes")
    print("\n--- first 400 characters ---")
    print(document[:400])
    print("...")

## MCP Servers for the Model

Our `pharmacy` server and this `fetch` server are **both just MCP servers**. You can list both sets of tools, hand them all to the model, and let the model choose:

```python
our_tools      = await pharmacy.list_tools()          # ours
their_listing  = await client.list_tools()            # somebody else's
all_tools      = our_tools + their_listing.tools      # one menu
```

The model does not know or care which is which. **That is what standardisation buys you.**

> ⚠️ **But be careful here.** ⚠️ 
>
> Connecting a server means **running somebody else's code with your permissions.** Before connecting one to anything real:
> * read what it actually does, and who maintains it
> * check what it can access — files, network, credentials
> * be very cautious with servers that can **write** anything
> * remember that content it fetches is *untrusted input* — a web page that says "ignore your instructions" is a real attack, not a hypothetical
>
> In a health environment, a third-party MCP server is a vendor dependency. Treat it like one.

---
# **What We Built Today**
---

**Four things to carry back:**

1. **MCP did not change what a tool is.** Still an ordinary Python function. It changed how tools are *described, discovered and called.*
2. **The schema now comes from your code.** Docstrings and type hints became the interface — what you publish and what you run cannot disagree.
3. **Discovery is the real feature.** We used the fetch server without reading its docs. We asked it what it could do, and it told us.
4. **Servers are independent processes.** Ours needed `mcp` 2.x, theirs needed 1.x, and they worked together anyway. That is what a protocol is for.

And still true from last session: **the model requests, your server executes.** MCP standardises the conversation — it does not hand over control. For member-facing actions, the human approval step remains yours to build.

> 🔐 **One caution to carry into work:** a third-party MCP server runs with your permissions. Vet it like any vendor dependency, and be especially careful with servers that can write.

---
# **Thank you !**
---

Author: *Aneetta Sara Shany*

Date: *2026 September 02, Wednesday*